In [2]:
import torch
import torch.nn as nn

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from tv_sim.tv_sim.core.utils.config import VehicleConfig

class TireLuGreRecurrentObserver(nn.Module):
    def __init__(self, cfg=VehicleConfig(),dt=0.001, hidden_dim=64):
        super().__init__()
        self.dt = dt
        self.hidden_dim = hidden_dim
        self.cfg=cfg
        # Input: [v_slip, F_z] (Operating conditions only)
        self.gru = nn.GRU(input_size=2, hidden_size=hidden_dim, batch_first=True)
        
        # Maps the GRU's "road context" 'h' to physical parameters
        self.param_head = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.Tanh(),
            nn.Linear(32, 5) # [s0, s1,  mu_k, mu_s_delta, v_s]
        )
        
        # Initialization
        self.init_weights()

    def forward(self, x, h_init=None, z_init=None):
        """
        x: [Batch, Seq_Len, 2] -> (v_slip, F_z)
        h_init: [1, Batch, Hidden] -> Initial GRU memory
        z_init: [Batch, 1] -> Initial bristle deflection
        """
        batch_size, seq_len, _ = x.shape
        
        # gru_out: [Batch, Seq, Hidden]
        gru_out, h_last = self.gru(x, h_init)
        
        # params_raw: [Batch, Seq, 6]
        params_raw = self.param_head(gru_out)
        
        # Decode parameters (Softplus for positivity)
        s0 = F.softplus(params_raw[:, :, 0]) * self.cfg.sigma0[0] # Stiffness
        s1 = F.softplus(params_raw[:, :, 1]) * self.cfg.sigma1[0] # Damping
        mu_k = F.softplus(params_raw[:, :, 2])       # Kinetic
        mu_s = mu_k + F.softplus(params_raw[:, :, 3])# Static (ensure > mu_k)
        v_s  = F.softplus(params_raw[:, :, 4]) + 0.1 # Stribeck Vel
        
        s2 = self.cfg.sigma2[0]  # Viscous fixed, not estimated

        # integration
        
        z_history = []
        F_history = []
        
        # Start with provided z_init or zeros
        if z_init is None:
            z_curr = torch.zeros(batch_size, device=x.device)
        else:
            z_curr = z_init.squeeze()

        for t in range(seq_len):
 
            v_t = x[:, t, 0]
            s0_t = s0[:, t]
            s1_t = s1[:, t]
            s2_t = self.cfg.sigma2[0]
            mu_k_t = mu_k[:, t]
            mu_s_t = mu_s[:, t]
            v_s_t  = v_s[:, t]
            Fz_t   = x[:, t, 1]
            
            #Stribeck Curve
    
            g_v = (mu_k_t + (mu_s_t - mu_k_t) * torch.exp(-(v_t/v_s_t)**2)) * Fz_t
            
            # Analytical Step (assume Piecewise linear ode)
            abs_v = torch.abs(v_t).clamp(min=1e-6)
            tau = g_v / (s0_t * abs_v + 1e-6)
            z_ss = torch.sign(v_t) * (g_v / (s0_t + 1e-6))
            
            z_next = z_ss + (z_curr - z_ss) * torch.exp(-self.dt / tau)
            
            # Compute Friction Force
            # F = s0*z + s1*dz + s2*v
            dz = (z_next - z_curr) / self.dt
            F_fric = s0_t * z_next + s1_t * dz + s2_t * v_t
            
            # Store
            z_history.append(z_next)
            F_history.append(F_fric)
            
            # Update state
            z_curr = z_next
            
        # Stack history: [Batch, Seq]
        z_out = torch.stack(z_history, dim=1)
        F_out = torch.stack(F_history, dim=1)
        
        return F_out, z_out, h_last, {
            "s0": s0, "mu_k": mu_k, "mu_s": mu_s
        }

    def init_weights(self):
        for name, param in self.gru.named_parameters():
            if 'weight_hh' in name:
                nn.init.orthogonal_(param)
            if 'bias' in name:
                nn.init.constant_(param, 0.0)
        
        # Bias initialization for easier convergence
        final_layer = self.param_head[-1]
        final_layer.bias.data[0].fill_(1.0) # s0
        final_layer.bias.data[2].fill_(0.5) # mu_k (FIXED INDEX)
        final_layer.bias.data[3].fill_(0.1) # mu_s_delta (FIXED INDEX)

In [ ]:

def criterion(F_pred, J, omega_dot_measured):
    Torque_pred = F_pred * r
    Torque_measured = J * omega_dot_measured
    return MSE(Torque_pred, Torque_measured)

In [ ]:
def steady_state_loss(v_slip, F_pred, mu_k, mu_s, v_s, F_z, s2):
    
    dv = torch.abs(v_slip[:, 1:] - v_slip[:, :-1])
    is_steady = dv < 0.05 # Threshold: 0.05 m/s^2 variation
    
    # F_ss = g(v) + s2*v
    g_v = (mu_k + (mu_s - mu_k) * torch.exp(-(v_slip/v_s)**2)) * F_z
    F_theoretical = g_v * torch.sign(v_slip) + s2 * v_slip
    
    # Apply Loss ONLY on steady frames

    loss = F.mse_loss(F_pred[:, 1:][is_steady], F_theoretical[:, 1:][is_steady])
    return loss

In [ ]:
def boundedness_loss(z_est, v_slip, s0, mu_k, mu_s, v_s, F_z):
    # Calculate the physical limit at every step
    g_v = (mu_k + (mu_s - mu_k) * torch.exp(-(v_slip/v_s)**2)) * F_z
    z_limit = g_v / (s0 + 1e-6)

    # We use ReLU so there is no penalty if z is inside the limit
    violation = F.relu(torch.abs(z_est) - z_limit)
    
    return torch.mean(violation**2)

In [ ]:
def smoothness_loss(parameters):
    # parameters: [Batch, Time, 6]
    # Penalize the difference between t and t-1
    diff = parameters[:, 1:, :] - parameters[:, :-1, :]
    return torch.mean(torch.abs(diff))